# Unified Offset-Free MPC Baseline

This notebook runs the canonical polymer baseline MPC workflow using `Polymer/Data` and `Polymer/Results`.

In [ ]:
from pathlib import Path
import os
import pickle

from systems.polymer.data_io import canonical_baseline_path
from utils.notebook_setup import prepare_polymer_notebook_env, print_notebook_summary

# User-editable run mode.
RUN_MODE = "nominal"  # "nominal" | "disturb"

# Plot/export controls.
STYLE_PROFILE = "hybrid"  # "hybrid" | "paper" | "debug"
SAVE_PDF = False

# Optional directory overrides. Leave as None to use Polymer/Data and Polymer/Results.
POLYMER_DATA_DIR_OVERRIDE = None
POLYMER_RESULTS_DIR_OVERRIDE = None

# Optional naming/path overrides.
RESULT_PREFIX_OVERRIDE = None
BASELINE_SAVE_PATH_OVERRIDE = None

# Optional run-size overrides. Leave as None to use the defaults from RUN_PROFILE_MAP.
N_TESTS_OVERRIDE = None
SET_POINTS_LEN_OVERRIDE = None
TEST_CYCLE_OVERRIDE = None
PLOT_START_EPISODE_OVERRIDE = None

REPO_ROOT, DATA_DIR, RESULT_DIR = prepare_polymer_notebook_env(
    data_dir_override=POLYMER_DATA_DIR_OVERRIDE,
    results_dir_override=POLYMER_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)

RUN_PROFILE_MAP = {
    "nominal": {
        "use_disturbance": False,
        "result_prefix": "mpc_offsetfree_nominal_unified",
        "save_path": canonical_baseline_path(REPO_ROOT, "nominal", data_override=POLYMER_DATA_DIR_OVERRIDE),
        "plot_start_episode": 1,
        "n_tests": 2,
        "set_points_len": 400,
        "test_cycle": [False, False],
        "nominal_qi": 108.0,
        "nominal_qs": 459.0,
        "nominal_ha": 1.05e6,
        "qi_change": 0.95,
        "qs_change": 1.05,
        "ha_change": 0.92,
    },
    "disturb": {
        "use_disturbance": True,
        "result_prefix": "mpc_offsetfree_disturb_unified",
        "save_path": canonical_baseline_path(REPO_ROOT, "disturb", data_override=POLYMER_DATA_DIR_OVERRIDE),
        "plot_start_episode": 2,
        "n_tests": 200,
        "set_points_len": 200,
        "test_cycle": [False, False, False, False, False],
        "nominal_qi": 108.0,
        "nominal_qs": 459.0,
        "nominal_ha": 1.05e6,
        "qi_change": 0.95,
        "qs_change": 1.05,
        "ha_change": 0.92,
    },
}
RUN_PROFILE = RUN_PROFILE_MAP[RUN_MODE]
RESULT_PREFIX = RESULT_PREFIX_OVERRIDE or RUN_PROFILE["result_prefix"]

print_notebook_summary(
    "Polymer baseline configuration",
    {
        "Repo root": REPO_ROOT,
        "Polymer data dir": DATA_DIR,
        "Polymer results": RESULT_DIR,
        "Run mode": RUN_MODE,
        "Baseline pickle": Path(BASELINE_SAVE_PATH_OVERRIDE).expanduser() if BASELINE_SAVE_PATH_OVERRIDE else RUN_PROFILE["save_path"],
    },
)


In [ ]:
import numpy as np

from Simulation.mpc import MpcSolverGeneral
from Simulation.system_functions import PolymerCSTR
from systems.polymer import (
    POLYMER_DELTA_T_HOURS,
    POLYMER_DESIGN_PARAMS,
    POLYMER_INPUT_BOUNDS,
    POLYMER_OBSERVER_POLES,
    POLYMER_RL_SETPOINTS_PHYS,
    POLYMER_SETPOINT_RANGE_PHYS,
    POLYMER_SS_INPUTS,
    POLYMER_SYSTEM_METADATA,
    POLYMER_SYSTEM_PARAMS,
    RL_REWARD_DEFAULTS,
    load_polymer_system_data,
)
from utils.helpers import apply_min_max
from utils.mpc_baseline_runner import run_offsetfree_mpc
from utils.plotting import plot_baseline_mpc_results
from utils.rewards import make_reward_fn_relative_QR


In [ ]:
# Build the polymer plant and load the canonical polymer data bundle.
system_params = POLYMER_SYSTEM_PARAMS.copy()
system_design_params = POLYMER_DESIGN_PARAMS.copy()
system_steady_state_inputs = POLYMER_SS_INPUTS.copy()
delta_t = POLYMER_DELTA_T_HOURS

cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states = {"ss_inputs": cstr.ss_inputs, "y_ss": cstr.y_ss}

setpoint_y = POLYMER_SETPOINT_RANGE_PHYS.copy()
u_min = POLYMER_INPUT_BOUNDS["u_min"].copy()
u_max = POLYMER_INPUT_BOUNDS["u_max"].copy()

system_data = load_polymer_system_data(
    REPO_ROOT,
    steady_states=steady_states,
    setpoint_y=setpoint_y,
    u_min=u_min,
    u_max=u_max,
    n_inputs=2,
    data_override=POLYMER_DATA_DIR_OVERRIDE,
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]

inputs_number = 2
y_sp_scenario = POLYMER_RL_SETPOINTS_PHYS.copy()
y_sp_scenario = apply_min_max(y_sp_scenario, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(
    steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:]
)


In [ ]:
# User-editable controller defaults.
# Observer setup: the shared baseline runner computes L internally from these poles.
n_tests = RUN_PROFILE["n_tests"] if N_TESTS_OVERRIDE is None else int(N_TESTS_OVERRIDE)
set_points_len = RUN_PROFILE["set_points_len"] if SET_POINTS_LEN_OVERRIDE is None else int(SET_POINTS_LEN_OVERRIDE)
TEST_CYCLE = list(RUN_PROFILE["test_cycle"]) if TEST_CYCLE_OVERRIDE is None else list(TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = RUN_PROFILE.get("plot_start_episode", 1) if PLOT_START_EPISODE_OVERRIDE is None else int(PLOT_START_EPISODE_OVERRIDE)
BASELINE_SAVE_PATH = Path(BASELINE_SAVE_PATH_OVERRIDE).expanduser() if BASELINE_SAVE_PATH_OVERRIDE else RUN_PROFILE["save_path"]

poles = POLYMER_OBSERVER_POLES.copy()

n_inputs = 2
predict_h = 9
cont_h = 3
Q1_penalty = 5.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0
USE_SHIFTED_MPC_WARM_START = False  # False matches legacy zero initialization; set True to reuse the previous MPC sequence.

u_ss = apply_min_max(steady_states["ss_inputs"], data_min[:n_inputs], data_max[:n_inputs])
b_min = apply_min_max(np.array([71.6, 78.0]), data_min[:n_inputs], data_max[:n_inputs]) - u_ss
b_max = apply_min_max(np.array([870.0, 670.0]), data_min[:n_inputs], data_max[:n_inputs]) - u_ss

MPC_obj = MpcSolverGeneral(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([Q1_penalty, Q2_penalty]),
    R_in=np.array([R1_penalty, R2_penalty]),
    NP=predict_h,
    NC=cont_h,
)

k_rel = np.array([0.003, 0.0003])
band_floor_phys = np.array([0.006, 0.07])
Q_diag = np.array([518.0, 90.0])
R_diag = np.array([90.0, 90.0])
reward_params, reward_fn = make_reward_fn_relative_QR(
    data_min,
    data_max,
    n_inputs=n_inputs,
    k_rel=k_rel,
    band_floor_phys=band_floor_phys,
    Q_diag=Q_diag,
    R_diag=R_diag,
    tau_frac=0.7,
    gamma_out=0.5,
    gamma_in=0.5,
    beta=7.0,
    gate="geom",
    lam_in=1.0,
    bonus_kind="exp",
    bonus_k=12.0,
    bonus_p=0.6,
    bonus_c=20.0,
)

print(reward_params)


print_notebook_summary(
    "Resolved baseline parameters",
    {
        "n_tests": n_tests,
        "set_points_len": set_points_len,
        "plot_start_episode": PLOT_START_EPISODE,
        "baseline_save_path": BASELINE_SAVE_PATH,
    },
)


In [ ]:
mpc_cfg = {
    "run_mode": RUN_MODE,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "warm_start": 0,
    "test_cycle": TEST_CYCLE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START,
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "nominal_qi": RUN_PROFILE["nominal_qi"],
    "nominal_qs": RUN_PROFILE["nominal_qs"],
    "nominal_ha": RUN_PROFILE["nominal_ha"],
    "qi_change": RUN_PROFILE["qi_change"],
    "qs_change": RUN_PROFILE["qs_change"],
    "ha_change": RUN_PROFILE["ha_change"],
    "b_min": b_min,
    "b_max": b_max,
}

runtime_ctx = {
    "system": PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t),
    "MPC_obj": MPC_obj,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "data_min": data_min,
    "data_max": data_max,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": poles,
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "system_metadata": POLYMER_SYSTEM_METADATA,
    "reward_params": reward_params,
    "system_metadata": POLYMER_SYSTEM_METADATA,
}

result_bundle = run_offsetfree_mpc(mpc_cfg=mpc_cfg, runtime_ctx=runtime_ctx)
result_bundle["reward_params"] = reward_params
result_bundle["run_profile"] = RUN_PROFILE

print(f"nFE: {result_bundle['nFE']}")
print(f"Avg reward samples: {len(result_bundle['avg_rewards'])}")


In [ ]:
out_dir = plot_baseline_mpc_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": PLOT_START_EPISODE,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
    },
)

legacy_payload = {
    "u_mpc": result_bundle["u"],
    "y_mpc": result_bundle["y"],
    "avg_rewards": result_bundle["avg_rewards"],
    "avg_rewards_mpc": result_bundle["avg_rewards_mpc"],
    "rewards_step": result_bundle["rewards_step"],
    "rewards_mpc": result_bundle["rewards_mpc"],
    "xhatdhat": result_bundle["xhatdhat"],
    "yhat": result_bundle["yhat"],
    "delta_y_storage": result_bundle["delta_y_storage"],
    "delta_u_storage": result_bundle["delta_u_storage"],
    "delat_u_storage": result_bundle["delta_u_storage"],
    "run_mode": result_bundle["run_mode"],
    "disturbance_profile": result_bundle["disturbance_profile"],
    "steady_states": result_bundle["steady_states"],
    "y_sp": result_bundle["y_sp"],
    "nFE": result_bundle["nFE"],
    "delta_t": result_bundle["delta_t"],
    "time_in_sub_episodes": result_bundle["time_in_sub_episodes"],
    "data_min": result_bundle["data_min"],
    "data_max": result_bundle["data_max"],
}

with open(BASELINE_SAVE_PATH, "wb") as file:
    pickle.dump(legacy_payload, file)

print(f"Plots saved to  : {out_dir}")
print(f"Baseline pickle : {RUN_PROFILE['save_path']}")
